# Data Quality Assessment
Qualitaetspruefung der Rohdaten und des integrierten Master-Datensatzes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

INTER_DIR = '../data/02_intermediate/'
PROCESSED_DIR = '../data/03_processed/'
DOCS_DIR = '../docs/'

# Alle relevanten Dateien laden
df_pks_raw = pd.read_csv(INTER_DIR + 'pks_cleaned.csv', sep=';')
df_pks_mapped = pd.read_csv(INTER_DIR + 'pks_mapped.csv', sep=';')
df_jus = pd.read_csv(INTER_DIR + 'justiz_mapped.csv', sep=';')
df_bev = pd.read_csv(INTER_DIR + 'bevoelkerung_mapped.csv', sep=';')
df_master = pd.read_csv(PROCESSED_DIR + 'master_kriminalitaet_1984_2024.csv', sep=';')
df_mapping = pd.read_csv(DOCS_DIR + 'mapping_table_autofilled_v2_haendisch.csv', sep=';')

print('Datensaetze geladen.')
print(f'  PKS cleaned:  {df_pks_raw.shape}')
print(f'  PKS mapped:   {df_pks_mapped.shape}')
print(f'  Justiz:       {df_jus.shape}')
print(f'  Bevoelkerung: {df_bev.shape}')
print(f'  Master:       {df_master.shape}')
print(f'  Mapping:      {df_mapping.shape}')

## 1. Vollstaendigkeit der Rohdaten
Fehlende Werte, Duplikate und Wertebereiche in den einzelnen Quellen pruefen.

In [ ]:
# Fehlende Werte pro Quelldatensatz
print('=== Fehlende Werte (NaN) pro Spalte ===')
for name, df in [('PKS cleaned', df_pks_raw), ('Justiz', df_jus), ('Bevoelkerung', df_bev)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print(f'  {name}: keine fehlenden Werte')
    else:
        print(f'  {name}:')
        for col, count in nulls.items():
            print(f'    {col}: {count} ({count/len(df)*100:.1f}%)')

# Duplikat-Check: gleiche (Jahr, Kategorie, Nationalitaet) darf nur einmal vorkommen
print('\n=== Duplikate ===')
dupes_jus = df_jus.duplicated(subset=['jahr', 'straftat_bezeichnung', 'nationalitaet']).sum()
dupes_pks = df_pks_mapped.duplicated(subset=['jahr', 'straftat_bezeichnung', 'nationalitaet']).sum()
dupes_bev = df_bev.duplicated(subset=['jahr', 'nationalitaet']).sum()
print(f'  Justiz:       {dupes_jus} Duplikate')
print(f'  PKS mapped:   {dupes_pks} Duplikate')
print(f'  Bevoelkerung: {dupes_bev} Duplikate')

# Wertebereiche
print('\n=== Wertebereiche (Jahr) ===')
for name, df, col in [('PKS', df_pks_raw, 'Jahr'), ('Justiz', df_jus, 'jahr'), ('Bevoelkerung', df_bev, 'jahr')]:
    print(f'  {name}: {df[col].min()} - {df[col].max()}')

## 2. Identity Resolution Coverage
Wie viele PKS-Delikte wurden erfolgreich einer Justiz-Kategorie zugeordnet?

In [ ]:
total_mappings = len(df_mapping)
mapped_ok = df_mapping[df_mapping['justiz_straftat_bezeichnung_mapped'] != 'NICHT GEFUNDEN - BITTE MANUELL EINTRAGEN']
not_mapped = total_mappings - len(mapped_ok)

print(f'Mapping-Tabelle: {total_mappings} PKS-Delikte')
print(f'  Zugeordnet:    {len(mapped_ok)} ({len(mapped_ok)/total_mappings*100:.1f}%)')
print(f'  Offen:         {not_mapped}')

# Verteilung der Zuordnungen auf Justiz-Kategorien
print(f'\nZuordnung auf {mapped_ok["justiz_straftat_bezeichnung_mapped"].nunique()} Justiz-Kategorien:')
counts = mapped_ok['justiz_straftat_bezeichnung_mapped'].value_counts()
display(counts.to_frame('anzahl_pks_delikte'))

## 3. Zeitreihen-Abdeckung im Master-Datensatz
Heatmap: Welche Kategorien haben in welchen Jahren Daten (Polizei vs. Justiz)?

In [ ]:
# Nur eine Nationalitaet fuer die Uebersicht (symmetrisch)
df_check = df_master[df_master['nationalitaet'] == 'Deutsch'].copy()

pks_coverage = df_check.pivot_table(
    index='straftat_bezeichnung', columns='jahr',
    values='tatverdaechtige', aggfunc='sum'
).fillna(0)

jus_coverage = df_check.pivot_table(
    index='straftat_bezeichnung', columns='jahr',
    values='verurteilte', aggfunc='sum'
).fillna(0)

# Binaer: 1 = Daten vorhanden, 0 = keine Daten
pks_bin = (pks_coverage > 0).astype(int)
jus_bin = (jus_coverage > 0).astype(int)

fig, axes = plt.subplots(2, 1, figsize=(18, 12), sharex=True)

sns.heatmap(pks_bin, cmap='Greens', cbar=False, linewidths=0.3, ax=axes[0])
axes[0].set_title('PKS (Tatverdaechtige): Datenabdeckung pro Jahr und Kategorie')
axes[0].set_ylabel('')

sns.heatmap(jus_bin, cmap='Blues', cbar=False, linewidths=0.3, ax=axes[1])
axes[1].set_title('Justiz (Verurteilte): Datenabdeckung pro Jahr und Kategorie')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# Fehlende PKS-Jahre
pks_missing_years = pks_bin.columns[pks_bin.sum(axis=0) == 0].tolist()
print(f'Jahre komplett ohne PKS-Daten: {pks_missing_years if pks_missing_years else "keine"}')

# Pro Kategorie: in wie vielen Jahren fehlen Polizeidaten?
pks_gaps = (pks_bin == 0).sum(axis=1)
pks_gaps = pks_gaps[pks_gaps > 0]
if len(pks_gaps) > 0:
    print(f'\nKategorien mit fehlenden PKS-Jahren:')
    for kat, n in pks_gaps.items():
        print(f'  {kat}: {n} Jahre ohne Daten')

## 4. Strukturbruch Wiedervereinigung
Sprung in den Bevoelkerungsdaten um 1990 sichtbar machen.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for nat in ['Deutsch', 'Nichtdeutsch']:
    subset = df_bev[df_bev['nationalitaet'] == nat]
    ax.plot(subset['jahr'], subset['bevoelkerung_gesamt'] / 1e6, marker='o', label=nat)

ax.axvline(x=1990, ls='--', color='gray', alpha=0.7, label='Wiedervereinigung')
ax.set_title('Bevoelkerungsentwicklung nach Nationalitaet')
ax.set_ylabel('Einwohner (Mio.)')
ax.set_xlabel('Jahr')
ax.legend()
plt.tight_layout()
plt.show()

# Sprung quantifizieren
for nat in ['Deutsch', 'Nichtdeutsch']:
    sub = df_bev[df_bev['nationalitaet'] == nat].sort_values('jahr')
    bev_89 = sub[sub['jahr'] == 1989]['bevoelkerung_gesamt'].values
    bev_92 = sub[sub['jahr'] == 1992]['bevoelkerung_gesamt'].values
    if len(bev_89) > 0 and len(bev_92) > 0:
        delta = (bev_92[0] - bev_89[0]) / bev_89[0] * 100
        print(f'{nat}: {bev_89[0]/1e6:.1f} Mio (1989) -> {bev_92[0]/1e6:.1f} Mio (1992), +{delta:.1f}%')

## 5. Master-Datensatz Zusammenfassung

In [ ]:
print('=== Master-Datensatz ===')
print(f'Zeilen:          {len(df_master)}')
print(f'Spalten:         {list(df_master.columns)}')
print(f'Kategorien:      {df_master["straftat_bezeichnung"].nunique()}')
print(f'Nationalitaeten: {df_master["nationalitaet"].unique().tolist()}')
print(f'Zeitraum:        {df_master["jahr"].min()} - {df_master["jahr"].max()}')

print(f'\n=== Fehlende Werte im Master ===')
for col in df_master.columns:
    n_null = df_master[col].isnull().sum()
    n_zero = (df_master[col] == 0).sum() if df_master[col].dtype in ['int64', 'float64'] else 0
    print(f'  {col}: {n_null} NaN, {n_zero} Nullen')

print(f'\n=== Deskriptive Statistik ===')
display(df_master.describe().round(1))